# Advanced ring kernel detection on difference images

**process**:
- Find the difference between neighbouring images.
- Normalise the difference images to min = 0. 
- Threshold marginally above the background value as detected by the first peak in the intensity histogram
- Convolve with ring kernel (of multiple sizes?)
- Mask out non-plaque areas of the convolution output.
- Peak detection over output.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from phageid.processing.layers import Difference, Threshold, Convolution, PeakFinder, AgglomeratePeaks, SubtractByFrame, Erosion, Dilation, LayerProduct
from phageid.processing.kernels import RingKernel
from phageid.utils import find_first_peak
from phageid.visualisation import video_image_sequence_with_scatter, video_image_sequence

In [ ]:
# load data
sample = "sample_04_05.npy"
# sample = "sample_04_06.npy"
# sample = "sample_02_06.npy"
# sample = "sample_03_08.npy"

file_path = Path(f"/home/finley/Work/RDS/projects/phageid/data/processed/splits_2025-01-31_15-27-52/{sample}")
data = np.load(file_path)
data_ = data.copy()

In [ ]:
masks = Threshold(thresh=lambda x: find_first_peak(x)*1.1, above=True)(data_.copy())
masks = Dilation(kernel_size=3, iterations=3)(masks)
masks = Erosion(kernel_size=3, iterations=3)(masks)
masks= masks[2:]
plt.imshow(masks[-5])
plt.colorbar()

In [ ]:
masks[-1].sum()

## Pre-processing
- Difference between adjacent images (separated by n_diff)
- Subtract the layer minimum from each difference layer
- Threshold by the fist peak in the histogram 

In [ ]:
difference_op = Difference(separation=2)
data_ = difference_op(data)
plt.imshow(data_[-2])

In [ ]:
data_ = SubtractByFrame(value=np.min)(data_)

In [ ]:
thresh_op = Threshold(thresh=lambda x: find_first_peak(x)+5, above=True, allow_equal=False)
data_ = thresh_op(data_)
plt.imshow(data_[-2])

In [ ]:
kernel = RingKernel(size=21, thickness=3, blur=1) 
kernel._kernel -= kernel._kernel.min()
kernel._kernel /= kernel._kernel.sum()

plt.imshow(kernel._kernel)
plt.colorbar()

In [ ]:
data_ = Convolution(kernel=kernel)(data_)
plt.imshow(data_[-2])

In [ ]:
data_= LayerProduct(values=masks[2:])(data_)
plt.imshow(data_[-3])

In [ ]:
points = PeakFinder(threshold_abs=0.65, footprint=np.ones((3, 3)))(data_)
# points = AgglomeratePeaks(min_distance=15)(points)

In [ ]:
video_image_sequence_with_scatter(data_, points, fps=5)